In [ ]:
# =============================================================================
#  CELL 0 - ISOLATED ENVIRONMENT SETUP (RUN THIS THEN JUMP TO CELL 3)
# =============================================================================
import subprocess
import sys
import os
from pathlib import Path

VENV_DIR = Path("vllm_env")
VENV_PYTHON = VENV_DIR / "bin" / "python"
VENV_PIP = VENV_DIR / "bin" / "pip"

print("Creating isolated virtual environment for vLLM...")
if not VENV_DIR.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)

print("Installing dependencies safely inside venv (this takes ~1-2 mins)...")
subprocess.run([
    str(VENV_PIP), "install", 
    "vllm>=0.8.0", "outlines", "requests", 
    "--quiet"
], check=True)

print(f"✓ Isolated environment ready at: {VENV_DIR.absolute()}")


In [ ]:
# =============================================================================
#  CELL 1 - SERVER SETUP & CROSS-TASK CACHE LINKING
# =============================================================================
import os
import hashlib
import json
import sys
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '1'
print(f"✓ Locked to GPU 1")

WORK_DIR = Path('.')  # <-- UPDATE THIS to your EditSVG clone path
os.chdir(WORK_DIR)
print(f"✓ Working directory: {os.getcwd()}")

print("\n=================================================================")
print(" LINKING VISUAL CACHE (CROSS-TASK SYNC & BUGFIX: v1 vs v2)")
print("=================================================================")
SOURCE_CACHE_PATH = Path('../vision_cache_extracted/vision_cache')  # <-- UPDATE THIS to your extracted zip path
native_cache_dir = Path('runs/vision_cache')
native_cache_dir.mkdir(parents=True, exist_ok=True)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from svgpatchlab.data.svgeditbench import parse_query, TASK_DIRECTORIES

converted = 0
for dom_file in sorted(SOURCE_CACHE_PATH.glob('*.json')):
    stem = dom_file.stem
    if not stem.startswith('change_color_'):
        continue
    emoji_id = stem.replace('change_color_', '').removesuffix('_dom')
    
    scene = json.loads(dom_file.read_text())
    # Broadcast this visual context to the hashes of ALL tasks for this emoji
    for task, tdir in TASK_DIRECTORIES.items():
        query_file = Path('SVGEditBench') / tdir / 'query' / f'{emoji_id}.txt'
        if not query_file.exists():
            continue
        
        try:
            _, source_svg = parse_query(query_file.read_text())
        except:
            continue
            
        svg_hash = hashlib.sha256(source_svg.encode()).hexdigest()
        hash_dir = native_cache_dir / svg_hash[:16]
        hash_dir.mkdir(exist_ok=True)
        
        for node in scene.get('nodes', []):
            vc = node.get('visual_context')
            if vc is None:
                continue
            out = hash_dir / f"{node['id']}-s384-v1.json"
            out.write_text(json.dumps(vc, indent=2, sort_keys=True))
            converted += 1

print(f'Cross-task cache linking complete: {converted} entries written across all tasks!')


✓ Locked to GPU 1
✓ Working directory: /home/trishita/workspace/EditSVG

 LINKING VISUAL CACHE (CROSS-TASK SYNC & BUGFIX: v1 vs v2)
Cross-task cache linking complete: 4728 entries written across all tasks!


In [ ]:
# =============================================================================
#  CELL 2 — DYNAMIC CONFIG PATCH (RESTORE SCHEMA & UPDATE TOKENS)
# =============================================================================
import json
from pathlib import Path

config_path = Path("configs/models/qwen3.5-4b-openai.json")
with open(config_path, "r") as f:
    config = json.load(f)


#EDIT CONFIGS BASED ON COMPUTE AVAILABLE
config["max_tokens"] = 4096
config["timeout"] = 900

# RESTORE MISSING SCHEMA
config["extra_body"] = {
    "response_format": {
      "type": "json_schema",
      "json_schema": {
        "name": "patch",
        "strict": True,
        "schema": {
          "type": "object",
          "properties": {
            "version": {"type": "integer"},
            "operations": {
              "type": "array",
              "items": {
                "type": "object",
                "properties": {
                  "op": {"type": "string", "enum": ["set_attributes"]},
                  "targets": {"type": "array", "items": {"type": "string"}},
                  "attributes": {"type": "object", "additionalProperties": {"type": "string"}}
                },
                "required": ["op", "targets", "attributes"],
                "additionalProperties": False
              }
            }
          },
          "required": ["version", "operations"],
          "additionalProperties": False
        }
      }
    }
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"\u2713 Dynamically increased max_tokens and restored schema in {config_path}")


✓ Dynamically increased max_tokens and restored schema in configs/models/qwen3.5-4b-openai.json


In [ ]:
# =============================================================================
#  CELL 3 - BOOT DOWNSTREAM 4B TEXT LLM
# =============================================================================
import os
import subprocess
import sys
from pathlib import Path
import urllib.request as _req
import time
VENV_PYTHON = Path("vllm_env") / "bin" / "python"

TEXT_MODEL = "Qwen/Qwen3.5-4B"
TEXT_PORT = 8000

print("=================================================================")
print(" STARTING DOWNSTREAM TEXT MODEL")
print("=================================================================")
vllm_env = os.environ.copy()
vllm_text_log = open("vllm_text_server.log", "w")
text_server_process = subprocess.Popen([
    str(VENV_PYTHON), "-m", "vllm.entrypoints.openai.api_server",
    "--model", TEXT_MODEL,
    "--port", str(TEXT_PORT),
    "--max-model-len", "32768",
    "--dtype", "bfloat16",
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", "0.90",
], stdout=vllm_text_log, stderr=subprocess.STDOUT, env=vllm_env)

POLL_TIMEOUT_SECONDS = 900
server_ready = False
for tick in range(POLL_TIMEOUT_SECONDS):
    if text_server_process.poll() is not None:
        log_tail = Path("vllm_text_server.log").read_text()[-10000:]
        raise RuntimeError(f"vLLM crashed! Log tail:\n{log_tail}")
    try:
        with _req.urlopen(f"http://localhost:{TEXT_PORT}/v1/models", timeout=2) as r:
            if r.status == 200:
                server_ready = True
                break
    except:
        pass
    time.sleep(1)

if not server_ready:
    raise RuntimeError(f"vLLM text server failed to start within {POLL_TIMEOUT_SECONDS} seconds.")

print(f"✓ 4B Text vLLM server online! (took {tick}s)")


In [ ]:
# =============================================================================
#  CELL 4 - SMOKE TEST EVALUATION
# =============================================================================
import json
import subprocess
from pathlib import Path

SMOKE_EVAL_DIR = Path("runs/smoke_test_vision")
CONFIG_PATH = "configs/experiments/skeleton_patch_vision_72b.json"

print("=================================================================")
print(" RUNNING SMOKE TEST (2 CASES PER TASK)")
print("=================================================================")

print("Installing rendering dependencies (cairosvg, Pillow)...")
subprocess.run(["python3", "-m", "pip", "install", "-e", ".[vision]", "--quiet"], check=True)

result = subprocess.run([
    "python3", "-m", "svgpatchlab.cli", "evaluate",
    "--config", CONFIG_PATH,
    "--limit-per-task", "2",
    "--output-dir", str(SMOKE_EVAL_DIR),
    "--render"
])

if result.returncode != 0:
    raise RuntimeError(f"Smoke test failed (exit code {result.returncode}).")

summary_file = SMOKE_EVAL_DIR / "summary.json"
if summary_file.exists():
    with open(summary_file, 'r') as f:
        summary = json.load(f)
    overall = summary.get("overall", {})
    valid_rate = overall.get('valid_output_rate', 0.0)
    exact_rate = overall.get('gold_patch_exact_rate', 0.0)
    print(f"\n[SMOKE TEST RESULTS] Valid Rate: {valid_rate*100:.1f}%, Exact Match: {exact_rate*100:.1f}%")
    print("-- Breakdown by Task --")
    for task_name, task_stats in summary.get("by_task", {}).items():
        t_rate = task_stats.get('gold_patch_exact_rate', 0.0)
        print(f"  {task_name}: Exact Match = {t_rate*100:.1f}%")
    if valid_rate < 0.50:
        raise RuntimeError(f"CRITICAL FAILURE: Only {valid_rate*100:.1f}% valid output rate. Check tokens or config!")
    print("\n✓ Smoke test passed! Proceeding to full evaluation...")


 RUNNING SMOKE TEST (2 CASES PER TASK)
Installing rendering dependencies (cairosvg, Pillow)...
{
  "architecture": "skeleton_patch",
  "by_task": {
    "change_color": {
      "cases": 2,
      "completion_tokens": 128,
      "gold_patch_exact_rate": 1.0,
      "mean_failure_aware_mse": 0.0,
      "mean_model_latency_seconds": 0.6970842177979648,
      "mean_mse_on_rendered": 0.0,
      "model_calls": 2,
      "prompt_tokens": 4400,
      "protected_geometry_rate": 1.0,
      "reference_structure_match_rate": 1.0,
      "valid_output_rate": 1.0
    },
    "crop_to_half": {
      "cases": 2,
      "completion_tokens": 142,
      "gold_patch_exact_rate": 1.0,
      "mean_failure_aware_mse": 0.0,
      "mean_model_latency_seconds": 0.37190294871106744,
      "mean_mse_on_rendered": 0.0,
      "model_calls": 2,
      "prompt_tokens": 4379,
      "protected_geometry_rate": 1.0,
      "reference_structure_match_rate": 1.0,
      "valid_output_rate": 1.0
    },
    "set_contour": {
      "cas

In [ ]:
# =============================================================================
#  CELL 5 - FINAL EVALUATION & RESULTS DOWNLOAD
# =============================================================================
import json
import subprocess
import zipfile
from pathlib import Path

FINAL_EVAL_DIR = Path("runs/skeleton_patch_vision")
FINAL_ZIP = "final_benchmark_results.zip"
CONFIG_PATH = "configs/experiments/skeleton_patch_vision_72b.json"

print("=================================================================")
print(" RUNNING FINAL DOWNSTREAM EVALUATION")
print("=================================================================")

result = subprocess.run([
    "python3", "-m", "svgpatchlab.cli", "evaluate",
    "--config", CONFIG_PATH,
    "--limit-per-task", "100",
    "--render"
])

if result.returncode != 0:
    raise RuntimeError(f"Evaluation failed (exit code {result.returncode}).")

summary_file = FINAL_EVAL_DIR / "summary.json"
if summary_file.exists():
    with open(summary_file, 'r') as f:
        summary = json.load(f)
        
    print("\n" + "="*65)
    print(" 🏆 FINAL EVALUATION RESULTS 🏆")
    print("="*65)
    overall = summary.get("overall", {})
    exact_rate = overall.get('gold_patch_exact_rate')
    exact_pct = (exact_rate * 100) if exact_rate is not None else 0.0
    valid_rate = overall.get('valid_output_rate')
    valid_pct = (valid_rate * 100) if valid_rate is not None else 0.0
    mse = overall.get('mean_failure_aware_mse')
    mse_str = f'{mse:.4f}' if mse is not None else 'N/A'
    
    print(f"Total Cases                 : {overall.get('cases')}")
    print(f"Gold Patch Exact Match Rate : {exact_pct:.2f}%")
    print(f"Mean Failure-Aware MSE      : {mse_str}")
    print(f"Valid Output Rate           : {valid_pct:.2f}%")
    print(f"Model Calls                 : {overall.get('model_calls', 0)}")
    print(f"Errors                      : {summary.get('errors', {})}")
    
    print("\n-- Breakdown by Task --")
    by_task = summary.get("by_task", {})
    for task_name, task_stats in by_task.items():
        t_rate = task_stats.get('gold_patch_exact_rate')
        t_pct = (t_rate * 100) if t_rate is not None else 0.0
        t_mse = task_stats.get('mean_failure_aware_mse')
        t_mse_str = f'{t_mse:.4f}' if t_mse is not None else 'N/A'
        print(f"  {task_name}: Exact Match = {t_pct:.2f}% | MSE = {t_mse_str}")

print(f"\nPackaging final results into '{FINAL_ZIP}'...")
with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in FINAL_EVAL_DIR.rglob("*"):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(FINAL_EVAL_DIR.parent))

try:
    text_server_process.terminate()
except:
    pass

print(f"\n✓ All done! The zip is located at: {Path(FINAL_ZIP).absolute()}")


 RUNNING FINAL DOWNSTREAM EVALUATION
{
  "architecture": "skeleton_patch",
  "by_task": {
    "change_color": {
      "cases": 100,
      "completion_tokens": 6625,
      "gold_patch_exact_rate": 0.76,
      "mean_failure_aware_mse": 0.0014581226474547292,
      "mean_model_latency_seconds": 0.34141610614955425,
      "mean_mse_on_rendered": 0.0014581226474547292,
      "model_calls": 100,
      "prompt_tokens": 284398,
      "protected_geometry_rate": 1.0,
      "reference_structure_match_rate": 0.76,
      "valid_output_rate": 1.0
    },
    "crop_to_half": {
      "cases": 100,
      "completion_tokens": 7100,
      "gold_patch_exact_rate": 1.0,
      "mean_failure_aware_mse": 0.0,
      "mean_model_latency_seconds": 0.36003165709786117,
      "mean_mse_on_rendered": 0.0,
      "model_calls": 100,
      "prompt_tokens": 283077,
      "protected_geometry_rate": 1.0,
      "reference_structure_match_rate": 1.0,
      "valid_output_rate": 1.0
    },
    "set_contour": {
      "cases": 